# Einfache BERT-Modelle

BERT-Modelle können eigenständig verwendet werden, um fehlende Wörter vorherzusagen.
Dieses Notebook zeigt im ersten Teil, wie dies mit dem ursprünglichen
BERT-Modell von Google funktioniert. Da sich die Abstraktionsebenen weiterentwickelt haben, verwenden wir dafür
`pipeline`.

Neben der Vorhersage fehlender Wörter eignen sich BERT-ähnliche Modelle auch für die Klassifikation. Der zweite Teil dieses Notebooks verwendet eine Textklassifikations-Pipeline, um eine (mehrsprachige) Sentimentanalyse durchzuführen.

## BERT models are Masked-Language models

They can guess missing words depending on the context they appear in!

In [ ]:
from transformers import pipeline

In [ ]:
unmasker = pipeline('fill-mask', model='google-bert/bert-base-uncased')

In [ ]:
unmasker("Machine [MASK] is a very powerful analysis technology.")

In [ ]:
unmasker("Der Chef der [MASK] Bundesregierung ist der deutsche Bundeskanzler.")

## Sentiment detection

As a simple example, BERT models can be used for sentiment detection.

Find more at https://huggingface.co/models?pipeline_tag=text-classification&sort=trending&search=sentiment


In [ ]:
# Load the classification pipeline with the specified model
sentiment = pipeline("text-classification", model="nlptown/bert-base-multilingual-uncased-sentiment", 
                     top_k=None)

In [ ]:
# Classify a new sentence
sentiment("I love online courses. I can learn a lot.")

In [ ]:
sentiment("I love online courses! I can learn a lot!")

In [ ]:
sentiment("I love online courses! This course is hot shit!")

In [ ]:
sentiment("I have to work so hard to learn this LLM stuff. Nothing works as expected!")

In [ ]:
sentiment("Online-Kurse gefallen mir sehr gut! Ich kann dabei viel lernen!")

In [ ]:
sentiment("Ganz klasse dieser Kurs, nix habe ich verstanden und nix hat funktioniert.")

In [ ]:
import pandas as pd
df = pd.read_json("10000_All_Beauty.json.xz")

In [ ]:
%%time
res = sentiment(df["text"].map(str).tolist(), batch_size=32, max_tokens=512, max_length=512)

In [ ]:
res[0:10]

In [ ]:
for i in range(1, 6):
    df[i] = None
df["avg"] = None
df["std"] = None

In [ ]:
import numpy as np
scores = []
for e in res:
    r = {}
    for ee in e:
        r[ee["label"]] = ee["score"]
    avg = r["1 star"] + 2*r["2 stars"] + 3*r["3 stars"] + 4*r["4 stars"] + 5*r["5 stars"]
    avg2 = r["1 star"] + 2*2*r["2 stars"] + 3*3*r["3 stars"] + 4*4*r["4 stars"] + 5*5*r["5 stars"]
    std = np.sqrt(avg2 - avg*avg)
    scores.append([r["1 star"], r["2 stars"], r["3 stars"], r["4 stars"], r["5 stars"], avg, std])

In [ ]:
df[[1, 2, 3, 4, 5, "avg", "std"]] = scores

In [ ]:
df

In [ ]:
df["std"].mean()

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
df[(df["rating"]==1) & (df["avg"] > 3)]

In [ ]:
df[(df["rating"]==5) & (df["avg"] < 2)][["text", "rating", "avg", "std"]]

# Zero-Shot-Klassifikation

In [ ]:
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

In [ ]:
sequence_to_classify = "Friedrich Merz ist der aktuelle Bundeskanzler."
candidate_labels = ["Politik", "Wirtschaft", "Unterhaltung", "Umwelt"]
output = classifier(sequence_to_classify, candidate_labels, multi_label=False)

In [ ]:
output

In [ ]:
!nvidia-smi